<a href="https://colab.research.google.com/github/sanrodmu29/Curso_Programacion_Cientifica/blob/main/ejercicios/taller-evaluativo-02-funciones-y-poo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Nombre:** Santiago Rodríguez Muñoz
**Fecha:** 2026-10-06

Calibración de un sensor de bajo costo de PM2.5 (SBC-A) frente a la red de monitoreo, integrando funciones (parámetros, retorno, `map`, `filter`, `lambda`) y programación orientada a objetos (clases, validación, encapsulamiento, `__repr__` y `__str__`).

## Ejercicio 1 — Calibrar una lectura

Se define `calibrar_lectura`, que aplica la recta de calibración `valor * pendiente + intercepto`. Los parámetros de calibración son opcionales (pendiente 1.0 e intercepto 0.0), de modo que sin ellos la lectura no se altera. La función entrega el resultado con `return`.

In [11]:
def calibrar_lectura(valor, pendiente=1.0, intercepto=0.0):
    """Aplica la recta de calibracion a una lectura y devuelve el valor calibrado."""
    return valor * pendiente + intercepto


sin_calibrar = calibrar_lectura(10.0)
calibrada = calibrar_lectura(10.0, 1.2, -0.5)
con_intercepto = calibrar_lectura(10.0, intercepto=2.0)

print("Sin calibrar:", sin_calibrar)
print("Calibrada (1.2, -0.5):", calibrada)
print("Solo con intercepto 2.0:", con_intercepto)

Sin calibrar: 10.0
Calibrada (1.2, -0.5): 11.5
Solo con intercepto 2.0: 12.0


## Ejercicio 2 — ¿Es utilizable la lectura?

Se declara la lista de lecturas crudas de SBC-A y se define `es_lectura_valida`, que devuelve `True` cuando la lectura no es el centinela -999.0 y está entre 0 y 500.0 inclusive. La función recibe el valor por parámetro y no usa variables globales.

In [12]:
lecturas_sbc_a = [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


def es_lectura_valida(valor):
    """Devuelve True si la lectura no es el centinela -999.0 y esta entre 0 y 500.0 inclusive."""
    return valor != -999.0 and 0 <= valor <= 500.0


print("25.0 ->", es_lectura_valida(25.0))
print("-999.0 ->", es_lectura_valida(-999.0))
print("600.0 ->", es_lectura_valida(600.0))
print("0.0 ->", es_lectura_valida(0.0))

25.0 -> True
-999.0 -> False
600.0 -> False
0.0 -> True


## Ejercicio 3 — Promedio de las lecturas útiles

Se define `promedio_valido`, que recorre la lista con un `for`, acumula solo las lecturas para las que `es_lectura_valida` devuelve `True` y devuelve su promedio. Si ninguna lectura es válida devuelve `None` en lugar de dividir entre cero. El acumulador y el contador son variables locales.



In [21]:
def promedio_valido(lecturas):
    """Devuelve el promedio de las lecturas validas, o None si no hay ninguna."""
    acumulado = 0.0
    contador = 0
    for lectura in lecturas:
        if es_lectura_valida(lectura):
            acumulado += lectura
            contador += 1
    if contador == 0:
        return None
    return acumulado / contador


promedio_sbc_a = promedio_valido(lecturas_sbc_a)

print("Promedio valido de SBC-A:", promedio_sbc_a)
print("Promedio sin lecturas validas:", promedio_valido([-999.0, -999.0]))

Promedio valido de SBC-A: 27.0
Promedio sin lecturas validas: None


## Ejercicio 4 — Calibrar la serie con `map`, `filter` y `lambda`

Con `filter` y `es_lectura_valida` se conservan solo las lecturas útiles; con `map` y una `lambda` se calibran usando la pendiente y el intercepto de SBC-A; y con `filter` y otra `lambda` se cuenta cuántas lecturas calibradas superan 35.4 (límite de la categoría Moderada). No se usan bucles `for` ni comprensiones de listas.

In [14]:
pendiente_sbc_a = 1.2
intercepto_sbc_a = -0.5

lecturas_utiles = filter(es_lectura_valida, lecturas_sbc_a)

lecturas_calibradas = list(
    map(lambda v: calibrar_lectura(v, pendiente_sbc_a, intercepto_sbc_a), lecturas_utiles)
)

excedencias = len(list(filter(lambda v: v > 35.4, lecturas_calibradas)))

print("Lecturas calibradas:", lecturas_calibradas)
print("Excedencias sobre 35.4:", excedencias)


Lecturas calibradas: [11.5, 29.5, 47.5, 9.1, 61.9]
Excedencias sobre 35.4: 2


## Ejercicio 5 — La clase `Lectura`

La clase `Lectura` convierte una lectura suelta en un objeto con `valor`, `unidad` y `codigo_sensor` (con valores por defecto en `__init__`). El método `a_microgramos` convierte a µg/m³ cuando la unidad es mg/m³, y `es_dato_faltante` detecta el centinela -999.0. La clase no valida: el centinela es un valor legítimo.


In [15]:
class Lectura:
    """Representa una lectura de PM2.5 con su valor, unidad y sensor de origen."""

    def __init__(self, valor, unidad="ug/m3", codigo_sensor="sin asignar"):
        """Guarda el valor, la unidad y el codigo del sensor de la lectura."""
        self.valor = valor
        self.unidad = unidad
        self.codigo_sensor = codigo_sensor

    def a_microgramos(self):
        """Devuelve el valor de la lectura expresado en ug/m3."""
        if self.unidad == "mg/m3":
            return self.valor * 1000
        return self.valor

    def es_dato_faltante(self):
        """Devuelve True si el valor es el centinela -999.0."""
        return self.valor == -999.0


lectura_campo = Lectura(0.0252, "mg/m3", "SBC-A")
lectura_faltante = Lectura(-999.0, codigo_sensor="SBC-A")
lectura_normal = Lectura(30.0)

print("lectura_campo en ug/m3:", lectura_campo.a_microgramos())
print("lectura_faltante es dato faltante:", lectura_faltante.es_dato_faltante())
print("lectura_campo es dato faltante:", lectura_campo.es_dato_faltante())
print("lectura_normal unidad:", lectura_normal.unidad)
print("lectura_normal codigo de sensor:", lectura_normal.codigo_sensor)


lectura_campo en ug/m3: 25.2
lectura_faltante es dato faltante: True
lectura_campo es dato faltante: False
lectura_normal unidad: ug/m3
lectura_normal codigo de sensor: sin asignar


## Ejercicio 6 — La clase `Sensor`: validación y encapsulamiento

`Sensor` impide que exista un sensor inválido: lanza `ValueError` con `raise` si el código es un texto vacío o si la pendiente es menor o igual a cero, antes de guardar cualquier dato. La pendiente y el intercepto se guardan como atributos internos (`_pendiente`, `_intercepto`). El método `calibrar` reutiliza `calibrar_lectura`, y `__repr__` y `__str__` hacen que el objeto se describa solo.


In [16]:
class Sensor:
    """Representa un sensor de bajo costo con su recta de calibracion."""

    def __init__(self, codigo, pendiente=1.0, intercepto=0.0):
        """Valida y guarda el codigo, la pendiente y el intercepto del sensor."""
        if codigo == "":
            raise ValueError("El codigo del sensor no puede ser un texto vacio.")
        if pendiente <= 0:
            raise ValueError(f"La pendiente debe ser mayor que cero, se recibio {pendiente}.")
        self.codigo = codigo
        self._pendiente = pendiente
        self._intercepto = intercepto

    def calibrar(self, valor):
        """Devuelve la lectura calibrada con la recta de este sensor."""
        return calibrar_lectura(valor, self._pendiente, self._intercepto)

    def __repr__(self):
        """Devuelve la representacion tecnica del sensor."""
        return (f"Sensor(codigo={self.codigo!r}, pendiente={self._pendiente}, "
                f"intercepto={self._intercepto})")

    def __str__(self):
        """Devuelve una descripcion legible del sensor en una linea."""
        return (f"Sensor {self.codigo}: pendiente {self._pendiente}, "
                f"intercepto {self._intercepto}")


sensor_a = Sensor("SBC-A", 1.2, -0.5)
sensor_b = Sensor("SBC-B")

print(sensor_a)
print([sensor_a, sensor_b])
print("Calibrar 10.0 con sensor_a:", sensor_a.calibrar(10.0))
print("Calibrar 10.0 con sensor_b:", sensor_b.calibrar(10.0))


Sensor SBC-A: pendiente 1.2, intercepto -0.5
[Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5), Sensor(codigo='SBC-B', pendiente=1.0, intercepto=0.0)]
Calibrar 10.0 con sensor_a: 11.5
Calibrar 10.0 con sensor_b: 10.0


Prueba de la validación (Ejercicio 6): se dejan comentadas las líneas que lanzan `ValueError` para que el notebook corra completo sin errores. Para comprobarlas a mano, quite el `#` de una línea, ejecute la celda y vuelva a comentarla (o borre esta celda) antes de entregar.

In [17]:
# Sensor("")      # debe fallar con ValueError (codigo vacio)
# Sensor("X", 0)  # debe fallar con ValueError (pendiente <= 0)

## Ejercicio 7 — La clase `Estacion`: objetos que contienen objetos

`Estacion` guarda objetos `Lectura` en una lista interna `_lecturas`. El método `agregar` añade una lectura, `valores` devuelve las concentraciones en µg/m³ (usando `a_microgramos()` de cada lectura) y `__repr__` describe la estación. Se carga la campaña de EST-04 recorriendo los pares (valor, unidad) con un `for`.


In [18]:
class Estacion:
    """Representa una estacion de monitoreo que guarda objetos Lectura."""

    def __init__(self, codigo, sector):
        """Guarda el codigo y el sector de la estacion y crea su lista de lecturas."""
        self.codigo = codigo
        self.sector = sector
        self._lecturas = []

    def agregar(self, lectura):
        """Agrega una Lectura a la estacion."""
        self._lecturas.append(lectura)

    def valores(self):
        """Devuelve la lista de concentraciones en ug/m3 de las lecturas guardadas."""
        return [lectura.a_microgramos() for lectura in self._lecturas]

    def __repr__(self):
        """Devuelve la representacion tecnica de la estacion."""
        return f"Estacion({self.codigo!r}, {self.sector!r}, lecturas={len(self._lecturas)})"


datos_campana = [
    (10.0, "ug/m3"), (-999.0, "ug/m3"), (0.025, "mg/m3"), (40.0, "ug/m3"),
    (-999.0, "ug/m3"), (8.0, "ug/m3"), (52.0, "ug/m3"), (600.0, "ug/m3"),
]

estacion_oriente = Estacion("EST-04", "Oriente")

for valor, unidad in datos_campana:
    estacion_oriente.agregar(Lectura(valor, unidad, "SBC-A"))

print(estacion_oriente)
print(estacion_oriente.valores())

Estacion('EST-04', 'Oriente', lecturas=8)
[10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]


Se define `categoria_pm25` (una sola cadena `if / elif / else` con la escala oficial) y la clase `InformeEstacion`, cuyos métodos reutilizan `es_lectura_valida`, `promedio_valido`, `categoria_pm25` y el método `calibrar` del sensor. `promedio_calibrado` filtra las lecturas útiles, las calibra con `map` y devuelve el promedio; `categoria` clasifica ese promedio; y `__str__` muestra el informe con el formato exacto pedido.

In [19]:
def categoria_pm25(concentracion):
    """Devuelve la categoria de calidad del aire para una concentracion de PM2.5."""
    if concentracion <= 12.0:
        return "Buena"
    elif concentracion <= 35.4:
        return "Moderada"
    elif concentracion <= 55.4:
        return "Dañina para grupos sensibles"
    else:
        return "Dañina para la salud"


class InformeEstacion:
    """Resume el estado de calidad del aire de una estacion con un sensor calibrado."""

    def __init__(self, estacion, sensor):
        """Guarda la estacion y el sensor como atributos internos."""
        self._estacion = estacion
        self._sensor = sensor

    def promedio_calibrado(self):
        """Devuelve el promedio de las lecturas validas de la estacion ya calibradas."""
        utiles = filter(es_lectura_valida, self._estacion.valores())
        calibradas = list(map(self._sensor.calibrar, utiles))
        return promedio_valido(calibradas)

    def categoria(self):
        """Devuelve la categoria de calidad del aire del promedio calibrado."""
        return categoria_pm25(self.promedio_calibrado())

    def __str__(self):
        """Devuelve el informe en una linea: estacion, promedio calibrado y categoria."""
        return (f"{self._estacion.codigo}: {round(self.promedio_calibrado(), 1)} ug/m3 "
                f"- {self.categoria()}")

    # Reto de extension (opcional, sin puntos)
    def excedencias(self, umbral=35.4):
        """Devuelve cuantas lecturas calibradas superan el umbral."""
        utiles = filter(es_lectura_valida, self._estacion.valores())
        calibradas = map(self._sensor.calibrar, utiles)
        return len(list(filter(lambda v: v > umbral, calibradas)))


informe = InformeEstacion(estacion_oriente, sensor_a)

print(informe)
print(informe.promedio_calibrado())
print("Excedencias (reto):", informe.excedencias())

EST-04: 31.9 ug/m3 - Moderada
31.9
Excedencias (reto): 2


## Celda de verificación

Se ejecuta al final, después de todo lo demás, sin modificarla.

In [20]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert abs(calibrar_lectura(10.0, 1.2, -0.5) - 11.5) < 1e-9, "revise el Ejercicio 1"
assert calibrar_lectura(10.0) == 10.0, "sin parámetros no debe alterar la lectura"
assert abs(con_intercepto - 12.0) < 1e-9, "revise con_intercepto del Ejercicio 1"
assert es_lectura_valida(25.0) and not es_lectura_valida(-999.0), "revise el Ejercicio 2"
assert not es_lectura_valida(600.0), "600.0 excede el rango del sensor"
assert abs(promedio_sbc_a - 27.0) < 1e-9, "revise el promedio del Ejercicio 3"
assert promedio_valido([-999.0, -999.0]) is None, "sin lecturas válidas debe devolver None"
assert len(lecturas_calibradas) == 5, "deberían quedar 5 lecturas calibradas"
assert abs(lecturas_calibradas[0] - 11.5) < 1e-9, "revise la calibración del Ejercicio 4"
assert excedencias == 2, "deberían ser 2 lecturas sobre 35.4"
assert abs(lectura_campo.a_microgramos() - 25.2) < 1e-9, "revise a_microgramos"
assert lectura_faltante.es_dato_faltante() and not lectura_campo.es_dato_faltante()
assert repr(sensor_a) == "Sensor(codigo='SBC-A', pendiente=1.2, intercepto=-0.5)", "revise __repr__ de Sensor"
assert hasattr(sensor_a, "_pendiente") and not hasattr(sensor_a, "pendiente"), "la pendiente debe ser interna"
assert abs(sensor_b.calibrar(10.0) - 10.0) < 1e-9, "revise los valores por defecto de Sensor"
assert repr(estacion_oriente) == "Estacion('EST-04', 'Oriente', lecturas=8)", "revise __repr__ de Estacion"
assert estacion_oriente.valores() == lecturas_sbc_a, "revise valores() del Ejercicio 7"
assert abs(informe.promedio_calibrado() - 31.9) < 1e-9, "revise el Ejercicio 8"
assert informe.categoria() == "Moderada", "revise categoria()"
assert str(informe) == "EST-04: 31.9 ug/m3 - Moderada", "revise __str__ de InformeEstacion"
print("Verificación completada sin errores.")

Verificación completada sin errores.
